# CUDA C Programming Lab Notebook
## List Ranking, Sparse Matrix Computation, and Convolution Kernel

This notebook contains three CUDA C programs designed for Jupyter Notebook / Google Colab:

1. **List Ranking** using pointer jumping
2. **Sparse Matrix Computation** using CSR and Sparse Matrix–Vector Multiplication
3. **1D Convolution Kernel** using CUDA

> A CUDA-capable NVIDIA GPU and `nvcc` are required. In Google Colab, select **Runtime → Change runtime type → GPU**.


## 0. Check CUDA Environment

In [ ]:
!nvidia-smi
!nvcc --version


# 1. List Ranking in CUDA

### Objective
For a linked list represented by `next[]`, calculate the distance/rank of every node from the end of the list.

Example:

```text
0 -> 1 -> 2 -> 3 -> 4 -> NULL
```

Ranks:

```text
0: 4, 1: 3, 2: 2, 3: 1, 4: 0
```

The kernel uses **pointer jumping** so that many list nodes are processed concurrently.


In [ ]:
%%writefile list_ranking.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define NULL_NODE -1

__global__ void listRankKernel(const int *next, int *rank,
                               int *next_new, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n) {
        int successor = next[i];

        if (successor != NULL_NODE) {
            rank[i] += rank[successor];
            next_new[i] = next[successor];
        } else {
            next_new[i] = NULL_NODE;
        }
    }
}

int main()
{
    const int n = 8;

    // 0 -> 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7 -> NULL
    int h_next[n] = {1, 2, 3, 4, 5, 6, 7, NULL_NODE};
    int h_rank[n];

    for (int i = 0; i < n; i++)
        h_rank[i] = (h_next[i] == NULL_NODE) ? 0 : 1;

    int *d_next, *d_rank, *d_next_new;

    cudaMalloc((void**)&d_next, n * sizeof(int));
    cudaMalloc((void**)&d_rank, n * sizeof(int));
    cudaMalloc((void**)&d_next_new, n * sizeof(int));

    cudaMemcpy(d_next, h_next, n * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_rank, h_rank, n * sizeof(int), cudaMemcpyHostToDevice);

    int blockSize = 256;
    int gridSize = (n + blockSize - 1) / blockSize;

    // Pointer jumping: approximately O(log n) rounds.
    for (int step = 1; step < n; step *= 2) {
        listRankKernel<<<gridSize, blockSize>>>(
            d_next, d_rank, d_next_new, n);

        cudaDeviceSynchronize();

        // Updated successor array for the next round.
        cudaMemcpy(d_next, d_next_new,
                   n * sizeof(int), cudaMemcpyDeviceToDevice);
    }

    cudaMemcpy(h_rank, d_rank,
               n * sizeof(int), cudaMemcpyDeviceToHost);

    printf("List Ranking Result:\n");
    for (int i = 0; i < n; i++)
        printf("Node %d -> Rank %d\n", i, h_rank[i]);

    cudaFree(d_next);
    cudaFree(d_rank);
    cudaFree(d_next_new);

    return 0;
}


In [ ]:
!nvcc list_ranking.cu -o list_ranking
!./list_ranking


### Expected output

```text
Node 0 -> Rank 7
Node 1 -> Rank 6
Node 2 -> Rank 5
Node 3 -> Rank 4
Node 4 -> Rank 3
Node 5 -> Rank 2
Node 6 -> Rank 1
Node 7 -> Rank 0
```

### Main CUDA concepts
- `threadIdx.x`: thread index inside a block
- `blockIdx.x`: block index
- `blockDim.x`: number of threads per block
- `<<<gridSize, blockSize>>>`: CUDA kernel launch
- Pointer jumping: repeatedly replace a node's successor with its successor's successor


# 2. Sparse Matrix Computation using CSR

### Objective

Represent a sparse matrix using **Compressed Sparse Row (CSR)** format and perform:

$$y = Ax$$

CSR stores:
- `values[]`: non-zero values
- `colIndex[]`: column of each non-zero value
- `rowPtr[]`: beginning/end position of each row

The CUDA kernel assigns **one thread to each matrix row**.


In [ ]:
%%writefile sparse_matrix.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void spmvCSR(int rows,
                        const int *rowPtr,
                        const int *colIndex,
                        const float *values,
                        const float *x,
                        float *y)
{
    int row = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < rows) {
        float sum = 0.0f;

        for (int j = rowPtr[row]; j < rowPtr[row + 1]; j++) {
            sum += values[j] * x[colIndex[j]];
        }

        y[row] = sum;
    }
}

int main()
{
    // Matrix:
    // [10  0  0 20]
    // [ 0 30  0  0]
    // [ 0  0 40 50]
    // [60  0  0 70]

    const int rows = 4;
    const int cols = 4;
    const int nnz = 7;

    int h_rowPtr[rows + 1] = {0, 2, 3, 5, 7};
    int h_colIndex[nnz] = {0, 3, 1, 2, 3, 0, 3};
    float h_values[nnz] = {10, 20, 30, 40, 50, 60, 70};

    float h_x[cols] = {1, 2, 3, 4};
    float h_y[rows] = {0};

    int *d_rowPtr, *d_colIndex;
    float *d_values, *d_x, *d_y;

    cudaMalloc((void**)&d_rowPtr, (rows + 1) * sizeof(int));
    cudaMalloc((void**)&d_colIndex, nnz * sizeof(int));
    cudaMalloc((void**)&d_values, nnz * sizeof(float));
    cudaMalloc((void**)&d_x, cols * sizeof(float));
    cudaMalloc((void**)&d_y, rows * sizeof(float));

    cudaMemcpy(d_rowPtr, h_rowPtr,
               (rows + 1) * sizeof(int),
               cudaMemcpyHostToDevice);

    cudaMemcpy(d_colIndex, h_colIndex,
               nnz * sizeof(int),
               cudaMemcpyHostToDevice);

    cudaMemcpy(d_values, h_values,
               nnz * sizeof(float),
               cudaMemcpyHostToDevice);

    cudaMemcpy(d_x, h_x,
               cols * sizeof(float),
               cudaMemcpyHostToDevice);

    int blockSize = 256;
    int gridSize = (rows + blockSize - 1) / blockSize;

    spmvCSR<<<gridSize, blockSize>>>(
        rows, d_rowPtr, d_colIndex, d_values, d_x, d_y);

    cudaDeviceSynchronize();

    cudaMemcpy(h_y, d_y,
               rows * sizeof(float),
               cudaMemcpyDeviceToHost);

    printf("Sparse Matrix-Vector Multiplication:\n");

    for (int i = 0; i < rows; i++)
        printf("y[%d] = %.2f\n", i, h_y[i]);

    cudaFree(d_rowPtr);
    cudaFree(d_colIndex);
    cudaFree(d_values);
    cudaFree(d_x);
    cudaFree(d_y);

    return 0;
}


In [ ]:
!nvcc sparse_matrix.cu -o sparse_matrix
!./sparse_matrix


### Expected output

```text
y[0] = 90.00
y[1] = 60.00
y[2] = 320.00
y[3] = 340.00
```

### CSR example

For the matrix:

```text
[10  0  0 20]
[ 0 30  0  0]
[ 0  0 40 50]
[60  0  0 70]
```

we store:

```text
values   = [10, 20, 30, 40, 50, 60, 70]
colIndex = [ 0,  3,  1,  2,  3,  0,  3]
rowPtr   = [ 0,  2,  3,  5,  7]
```

Only the 7 non-zero elements are stored instead of all 16 elements.


# 3. Convolution Kernel in CUDA

### Objective

Implement a simple **1D convolution** in CUDA.

Each CUDA thread computes one output element:

$$y[i] = \sum_j x[i+j-r]k[j]$$

where `r = kernel_size / 2`.

The implementation uses **zero padding** at the boundaries.


In [ ]:
%%writefile convolution.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void convolution1D(const float *input,
                              const float *kernel,
                              float *output,
                              int n,
                              int kSize)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    int radius = kSize / 2;

    if (i < n) {
        float sum = 0.0f;

        for (int j = 0; j < kSize; j++) {
            int index = i + j - radius;

            // Zero padding at boundaries.
            if (index >= 0 && index < n)
                sum += input[index] * kernel[j];
        }

        output[i] = sum;
    }
}

int main()
{
    const int n = 8;
    const int kSize = 3;

    float h_input[n] = {1, 2, 3, 4, 5, 6, 7, 8};
    float h_kernel[kSize] = {1, 2, 1};
    float h_output[n] = {0};

    float *d_input, *d_kernel, *d_output;

    cudaMalloc((void**)&d_input, n * sizeof(float));
    cudaMalloc((void**)&d_kernel, kSize * sizeof(float));
    cudaMalloc((void**)&d_output, n * sizeof(float));

    cudaMemcpy(d_input, h_input,
               n * sizeof(float),
               cudaMemcpyHostToDevice);

    cudaMemcpy(d_kernel, h_kernel,
               kSize * sizeof(float),
               cudaMemcpyHostToDevice);

    int blockSize = 256;
    int gridSize = (n + blockSize - 1) / blockSize;

    convolution1D<<<gridSize, blockSize>>>(
        d_input, d_kernel, d_output, n, kSize);

    cudaDeviceSynchronize();

    cudaMemcpy(h_output, d_output,
               n * sizeof(float),
               cudaMemcpyDeviceToHost);

    printf("Convolution Result:\n");

    for (int i = 0; i < n; i++)
        printf("output[%d] = %.2f\n", i, h_output[i]);

    cudaFree(d_input);
    cudaFree(d_kernel);
    cudaFree(d_output);

    return 0;
}


In [ ]:
!nvcc convolution.cu -o convolution
!./convolution


### Expected output

```text
output[0] = 4.00
output[1] = 8.00
output[2] = 12.00
output[3] = 16.00
output[4] = 20.00
output[5] = 24.00
output[6] = 28.00
output[7] = 23.00
```

The filter is:

```text
[1 2 1]
```

For example, for the middle element:

```text
output[3] = 3(1) + 4(2) + 5(1)
          = 16
```


# 4. Summary

| Program | Data structure / operation | CUDA parallelization |
|---|---|---|
| List Ranking | Linked list | One thread per node |
| Sparse Matrix | CSR + SpMV | One thread per row |
| Convolution | 1D convolution | One thread per output |

## Common CUDA workflow

```text
CPU
 |
 | cudaMalloc()
 v
GPU Memory
 |
 | cudaMemcpy()
 v
CUDA Kernel <<<grid, block>>>
 |
 | cudaDeviceSynchronize()
 v
GPU Result
 |
 | cudaMemcpy()
 v
CPU
```

## Suggested laboratory extensions

1. Increase list size from 8 to 1,024 nodes.
2. Generate a random sparse matrix and compare CSR with dense storage.
3. Implement 2D image convolution.
4. Use CUDA shared memory to optimize convolution.
5. Compare execution time for different block sizes: 64, 128, 256, and 512.
6. Add CUDA error checking using `cudaGetLastError()`.
7. Compare GPU and CPU execution times.
